In [6]:
import os

print(os.listdir('/kaggle/input/competitions/rsna-knee-abnormality-detection'))

['test_series.csv', 'train_series.csv', 'sample_submission.csv', 'test_series', 'train_series', 'train.csv', 'test.csv']


In [7]:
import pandas as pd

DATA_DIR = '/kaggle/input/competitions/rsna-knee-abnormality-detection'

train = pd.read_csv(f'{DATA_DIR}/train.csv')
series = pd.read_csv(f'{DATA_DIR}/train_series.csv')

print("Number of studies (train.csv rows):", train.shape[0])
print("Number of series (train_series.csv rows):", series.shape[0])
train.head()

Number of studies (train.csv rows): 4407
Number of series (train_series.csv rows): 24371


,StudyInstanceUID,Report,ACL,MCL,Medial Meniscus,Lateral Meniscus,Medial OA,Lateral OA,PF OA,Effusion,Synovitis,Baker's,Contusion,Fracture
0,1.2.826.0.1.3680043.8.498.10004873229099053869...,Técnica: RMN de la rodilla. Resultados: Rotura...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,1.2.826.0.1.3680043.8.498.10004945927472656027...,[DATE]: * MR Knie Rechts 15ch AA Klinische Inl...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,1.2.826.0.1.3680043.8.498.10009278692606631573...,Hallazgos:\nNo hay alteraciones en significati...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
3,1.2.826.0.1.3680043.8.498.10009639203170750274...,"In the medial compartment, the meniscus is no...",NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
4,1.2.826.0.1.3680043.8.498.10013663742400736029...,CONSTATATIONS :\n\nFractures :\nAucune.\n\nAli...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


In [8]:
label_cols = ['ACL', 'MCL', 'Medial Meniscus', 'Lateral Meniscus', 'Medial OA',
              'Lateral OA', 'PF OA', 'Effusion', 'Synovitis', "Baker's",
              'Contusion', 'Fracture']

# A study counts as "labeled" if at least one of the 12 columns is filled in
has_labels = train[label_cols].notna().any(axis=1)

n_labeled = has_labels.sum()
n_total = len(train)

print(f"Studies with at least one real label: {n_labeled} out of {n_total} ({n_labeled/n_total:.1%})")
print(f"Studies with only a report, no labels: {n_total - n_labeled} ({(n_total-n_labeled)/n_total:.1%})")

Studies with at least one real label: 58 out of 4407 (1.3%)
Studies with only a report, no labels: 4349 (98.7%)


In [12]:
train['report_length'] = train['Report'].str.len()
print(train['report_length'].describe())

for i in range(3):
    print(f"--- Report {i} ---")
    print(train['Report'].iloc[i])
    print()

count    4407.000000
mean     1097.905605
std       693.948925
min        52.000000
25%       587.500000
50%       977.000000
75%      1459.500000
max      4743.000000
Name: report_length, dtype: float64
--- Report 0 ---
Técnica: RMN de la rodilla. Resultados: Rotura de menisco interno. Signo de necrosis avascular subcondral en el cóndilo femoral medial. Artrosis femorotibial medial. Derrame. . Impresión: Rotura de menisco interno. Signo de necrosis avascular subcondral en el cóndilo femoral medial. Artrosis femorotibial medial. Derrame.

--- Report 1 ---
[DATE]: * MR Knie Rechts 15ch AA Klinische Inlichtingen: [DATE]. Diagnostische vraagstellling: Meniscusscheur/mediaal? Scanprotocol (DRB) : sag intermediair gewogen seq zonder en met fs, ax/ cor pd gewogen seq fs, cor T1 gewogen seq Bevindingen:

--- Report 2 ---
Hallazgos:
No hay alteraciones en significativas de la médula ósea.
Ligamentos cruzados y colaterales dentro de límites normales.
Aumento de señal del cuerno posterior del me

In [13]:
!pip install langdetect -q

from langdetect import detect, DetectorFactory
DetectorFactory.seed = 0  # makes detection results reproducible

def safe_detect(text):
    try:
        return detect(text)
    except Exception:
        return 'unknown'

train['report_lang'] = train['Report'].apply(safe_detect)
print(train['report_lang'].value_counts())

report_lang
en    1736
es     682
tr     546
hr     406
el     321
de     262
bg     220
nl     153
fr      81
Name: count, dtype: int64
